[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S07/2_Practica/para_casa/S07_04_roc_pr_desbalance.ipynb)

# S07 · 04 — Curvas ROC y PR, y qué hacer con clases desbalanceadas

Sprint 7 · M3 (UD 3.4)

**Contexto TurisData.** Imagina un alojamiento con clientes muy fieles donde **solo el 6 %** de las reservas se cancela. Un modelo que diga siempre «no cancelará» acertaría el 94 %… y sería completamente inútil. Cuando el evento que nos importa es **raro** (cancelaciones, fraudes, averías), la exactitud engaña y hay que usar otras herramientas.

**Este cuaderno tiene 6 ejercicios y un mini-reto.** El ejercicio 2 es el mismo que el del cuaderno «en clase» (allí, el 1).

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils import resample
from sklearn.base import clone
from sklearn.metrics import (roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
                             precision_score, recall_score, accuracy_score)

warnings.filterwarnings("ignore", category=UserWarning)
SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["temporada_alta"] = df["fecha_llegada"].dt.month.isin([1, 2, 3, 7, 8, 12]).astype(int)

# Escenario simulado: conservamos todas las no canceladas y una parte de las canceladas (≈ 6 %)
neg = df[df["cancelada"] == 0]
pos = df[df["cancelada"] == 1].sample(n=int(len(neg) * 0.064), random_state=SEMILLA)
raro = pd.concat([neg, pos]).sample(frac=1, random_state=SEMILLA).reset_index(drop=True)
print(f"Reservas: {len(raro)} · canceladas: {raro['cancelada'].sum()} ({raro['cancelada'].mean():.1%})")

columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "temporada_alta"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = raro[columnas_num + columnas_cat], raro["cancelada"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=SEMILLA, stratify=y)

prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), columnas_num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), columnas_cat),
])

def nuevo_modelo(**kwargs):
    # clone(prep): cada modelo necesita su propia copia del preprocesado (si no, se pisarían entre sí)
    return Pipeline([("prep", clone(prep)), ("clf", LogisticRegression(max_iter=1000, **kwargs))])
print("Listo ✔")

### ✏️ A2 · EJERCICIO 1 de 6 — El modelo inútil que acierta el 94 %

**Tu turno:** Entrena un `DummyClassifier(strategy="most_frequent")` con train. Guarda en `acc_tonto` su exactitud en test y en `recall_tonto` su recall en test (`recall_score`). ¿Qué te dice?

**Debe salir:** `Exactitud del modelo inútil: 94.0% · recall: 0%` y ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A2).

1. ¿Qué exactitud y qué recall tiene el modelo tonto?
2. ¿Por qué acierta tanto sin saber nada?
3. ¿Qué te dice sobre usar la exactitud con clases raras?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A2, **después de intentarlo**.

In [ ]:
# pista: DummyClassifier(strategy="most_frequent").fit(X_train, y_train); recall_score(y_test, pred)
tonto = ...
pred_tonto = ...
acc_tonto = ...
recall_tonto = ...

assert acc_tonto > 0.9, "El modelo que dice siempre «no cancela» acierta más del 90 %"
assert recall_tonto == 0, "…pero no detecta ninguna cancelación (recall = 0)"
print(f"Exactitud del modelo inútil: {acc_tonto:.1%} · recall: {recall_tonto:.0%}")
print("✔ Ejercicio 1 correcto")

### 👀 A3 · EJEMPLO — Entrenar el modelo base y sacar sus probabilidades

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A3).

1. ¿Cuántas reservas hay en test y cuántas canceladas?
2. ¿Qué contiene `proba_base`?
3. Con solo 62 positivas en test, ¿qué cuidado hay que tener al interpretar las métricas?

In [ ]:
base = nuevo_modelo().fit(X_train, y_train)
proba_base = base.predict_proba(X_test)[:, 1]
print("Reservas de test:", len(y_test), "· canceladas:", int(y_test.sum()))

### ✏️ A4 · EJERCICIO 2 de 6 — Calcular la curva ROC y el AUC

**Tu turno:** Con `y_test` y `proba_base` calcula `fpr, tpr, _ = roc_curve(...)` y `auc = roc_auc_score(...)`.

**Debe salir:** ✔ Ejercicio 2 correcto y un gráfico con la curva ROC (AUC ≈ 0,77)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A4).

1. ¿Cuánto vale el AUC?
2. ¿Cómo se interpreta un AUC de 0,77?
3. ¿Qué indica que la curva quede por encima de la diagonal?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A4, **después de intentarlo**.

In [ ]:
# pista: roc_curve(y_test, proba_base) devuelve tres arrays; roc_auc_score(y_test, proba_base)
fpr, tpr, _ = ...
auc = ...

assert len(fpr) == len(tpr) and fpr[0] == 0 and tpr[-1] == 1, "fpr y tpr deben ser la curva ROC completa"
assert 0.6 < auc < 0.95, "El AUC de este modelo debería estar entre 0,6 y 0,95"

plt.figure(figsize=(5.5, 5))
plt.plot(fpr, tpr, color="#2a6f97", lw=2, label=f"Regresión logística (AUC = {auc:.2f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Azar (AUC = 0,50)")
plt.title("Curva ROC")
plt.xlabel("Tasa de falsos positivos (falsas alarmas)")
plt.ylabel("Tasa de verdaderos positivos (recall)")
plt.legend(loc="lower right")
plt.show()
print("✔ Ejercicio 2 correcto")

### 👀 A5 · EJEMPLO — Curva precisión–recall

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A5).

1. ¿Cuánto valen la prevalencia y la AP?
2. ¿Qué representa la línea gris?
3. ¿Por qué la precisión cae al aumentar el recall?

In [ ]:
prec, rec, _ = precision_recall_curve(y_test, proba_base)
ap = average_precision_score(y_test, proba_base)
prevalencia = y_test.mean()
plt.figure(figsize=(5.5, 5))
plt.plot(rec, prec, color="#2a6f97", lw=2, label=f"Regresión logística (AP = {ap:.2f})")
plt.axhline(prevalencia, ls="--", color="gray", label=f"Azar (= prevalencia {prevalencia:.2f})")
plt.title("Curva precisión–recall")
plt.xlabel("Recall (cancelaciones detectadas)")
plt.ylabel("Precisión (alarmas acertadas)")
plt.legend()
plt.show()

### ✏️ A6 · EJERCICIO 3 de 6 — Cuánto mejora la AP al azar

**Tu turno:** Guarda en `ap_base` la *average precision* de `proba_base`, y en `veces_mejor_que_azar` cuántas veces mejor es que la prevalencia (`ap_base / prevalencia`).

**Debe salir:** `AP = 0.24: el modelo es 4.0 veces mejor que el azar en precisión.` y ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A6).

1. ¿Cuánto valen `ap_base` y `veces_mejor_que_azar`?
2. ¿Qué significa «4 veces mejor que el azar»?
3. El AUC (0,77) parece bueno y la AP (0,24) parece pobre. ¿Por qué?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A6, **después de intentarlo**.

In [ ]:
# pista: average_precision_score(y_test, proba_base); prevalencia ya está calculada
ap_base = ...
veces_mejor_que_azar = ...

assert 0 < ap_base < 1
assert veces_mejor_que_azar > 1.5, "El modelo debería ser claramente mejor que el azar"
print(f"AP = {ap_base:.2f}: el modelo es {veces_mejor_que_azar:.1f} veces mejor que el azar en precisión.")
print("✔ Ejercicio 3 correcto")

### ✏️ A7 · EJERCICIO 4 de 6 — Pesos por clase

**Tu turno:** Crea `con_pesos = nuevo_modelo(class_weight="balanced")`, ajústalo con train y guarda en `recall_base` y `recall_pesos` el recall en test (umbral 0,5) del modelo base y del ponderado. Guarda también `precision_pesos`.

**Debe salir:** `Recall: 3% → 73% · precisión del ponderado: 14%` y ✔ Ejercicio 4 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A7).

1. ¿Cómo cambia el recall y qué precisión tiene el modelo ponderado?
2. ¿Qué hace `class_weight="balanced"`?
3. ¿Qué se paga a cambio de detectar el 73 %?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A7, **después de intentarlo**.

In [ ]:
# pista: nuevo_modelo(class_weight="balanced").fit(...); recall_score(y_test, modelo.predict(X_test))
con_pesos = ...
recall_base = ...
recall_pesos = ...
precision_pesos = ...

assert recall_pesos > recall_base, "Con pesos por clase el recall debería subir"
assert recall_pesos > 0.5, "El modelo ponderado debería detectar más de la mitad de las cancelaciones"
print(f"Recall: {recall_base:.0%} → {recall_pesos:.0%} · precisión del ponderado: {precision_pesos:.0%}")
print("✔ Ejercicio 4 correcto")

### ✏️ A8 · EJERCICIO 5 de 6 — Submuestreo de la clase mayoritaria

**Tu turno:** Crea `X_sub, y_sub` a partir **solo de train**: todas las positivas y una muestra aleatoria de negativas del **doble** de tamaño (`resample(..., replace=False, n_samples=..., random_state=SEMILLA)`). Entrena `con_submuestreo = nuevo_modelo()` con esos datos.

**Debe salir:** ✔ Ejercicio 5 correcto (`X_sub` tiene 435 filas: 145 positivas y 290 negativas)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A8).

1. ¿Cuántas filas tiene `X_sub` y cómo se reparten?
2. ¿Por qué el recall del submuestreo (≈ 0,45) es menor que el de los pesos (0,73)?
3. ¿Qué inconveniente tiene el submuestreo?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A8, **después de intentarlo**.

In [ ]:
# pista: separa positivas y negativas de train, resample() de las negativas y concatena
X_sub = ...
y_sub = ...
con_submuestreo = ...

assert y_sub.sum() * 2 == (y_sub == 0).sum(), "Debe haber el doble de negativas que de positivas"
assert len(X_sub) < len(X_train) / 3, "El submuestreo reduce mucho el tamaño del entrenamiento"
assert recall_score(y_test, con_submuestreo.predict(X_test)) > recall_base, "El submuestreo debería subir el recall"
print("✔ Ejercicio 5 correcto")

### ✏️ A9 · EJERCICIO 6 de 6 — Umbral para detectar al menos el 70 %

**Tu turno:** El gerente pide **detectar al menos el 70 % de las cancelaciones** con la mayor precisión posible. Con las probabilidades de validación cruzada de `base` sobre train (`proba_cv`), usa `precision_recall_curve` y guarda en `umbral_70` el **mayor umbral** cuyo recall sea ≥ 0,70. Después mide en `recall_test_70` y `precision_test_70` cómo funciona ese umbral en test.

**Debe salir:** `Umbral 0.048: recall en test 82%, precisión 12%` y ✔ Ejercicio 6 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A9).

1. ¿Qué umbral sale y qué recall y precisión da en test?
2. ¿Por qué el umbral es tan bajo (0,048) y no 0,5?
3. El objetivo era el 70 % y en test sale el 82 %. ¿Debemos fiarnos?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A9, **después de intentarlo**.

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=SEMILLA)
proba_cv = cross_val_predict(nuevo_modelo(), X_train, y_train, cv=cv, method="predict_proba")[:, 1]
# pista: precision_recall_curve devuelve (precision, recall, umbrales); recall tiene un elemento más
umbral_70 = ...
recall_test_70 = ...
precision_test_70 = ...

assert 0 < umbral_70 < 0.5, "Para alcanzar recall 0,70 con clases raras el umbral debe ser bajo (< 0,5)"
assert recall_test_70 > 0.5, "En test el recall debería aproximarse al objetivo del 70 %"
print(f"Umbral {umbral_70:.3f}: recall en test {recall_test_70:.0%}, precisión {precision_test_70:.0%}")
print("✔ Ejercicio 6 correcto")

### ✏️ A10 · MINI-RETO — Comparativa de las tres estrategias

**Tu turno:** Crea `comparativa`, un DataFrame con una fila por estrategia (`"base"`, `"pesos"`, `"submuestreo"`) y las columnas `AUC`, `AP` (average precision, con probabilidades) y `recall` y `precision` (con umbral 0,5), todo en test. Redondea a 3 decimales. Guarda en `ganadora_ap` el nombre de la estrategia con mayor AP.

**Debe salir:** ✔ Mini-reto correcto (y la tabla `comparativa`)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A10).

1. ¿Qué estrategia gana en AP y cuánta ventaja tiene?
2. ¿Qué cambia mucho entre las tres estrategias?
3. ¿Por qué el AUC y la AP casi no cambian?

In [ ]:
# pista: recorre un dict {nombre: modelo}; predict_proba para AUC/AP y predict para recall/precisión
comparativa = ...
ganadora_ap = ...

assert list(comparativa.index) == ["base", "pesos", "submuestreo"]
assert {"AUC", "AP", "recall", "precision"} <= set(comparativa.columns)
assert ganadora_ap in comparativa.index
display(comparativa)
print("✔ Mini-reto correcto")

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas y el escenario de clase rara

> **Nota honesta:** `reservas_turisdata.csv` tiene un 46 % de cancelaciones. Para este laboratorio **simulamos** un escenario raro conservando solo una parte de las canceladas. Es un recurso didáctico: en un proyecto real no se manipula la tasa de la población de test.

- **Qué hace:**
    - Carga las herramientas: `pandas`, `numpy`, `matplotlib`, `seaborn` y, de `scikit-learn`, la regresión logística, el modelo tonto, el preprocesado, `resample` y las métricas ROC y PR.
    - `dato(...)` busca un archivo de datos del curso y, si no está, lo descarga del repositorio.
    - **Fabrica el escenario raro**: conserva todas las reservas no canceladas y solo una muestra de las canceladas (`pos`), de modo que queden ≈ 6 % (la tabla `raro`).
    - Separa `X` e `y`, reparte train y test (70 % / 30 %, estratificado) y define `prep` (el preprocesado) y `nuevo_modelo(**kwargs)`, que fabrica un pipeline nuevo cada vez (con su propia copia del preprocesado).
- **Qué debe salir:**
    - `Reservas: 3453 · canceladas: 207 (6.0%)` y `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. La paradoja de la exactitud

#### A2 · Ejercicio 1 — El modelo inútil que acierta el 94 %

- **Qué hace:**
    - `DummyClassifier(strategy="most_frequent")` es el modelo tonto: predice siempre la clase más frecuente (aquí, «no cancela»). `.fit(...)` solo cuenta cuál es.
    - `accuracy_score` y `recall_score` comparan sus predicciones con la realidad (`y_test`): exactitud y recall.
- **Qué debe salir:**
    - `Exactitud del modelo inútil: 94.0% · recall: 0%`.
    - `✔ Ejercicio 1 correcto`.
- **Si falla:**
    - `El modelo que dice siempre «no cancela» acierta más del 90 %` → no has puntuado con `y_test` y las predicciones de `X_test`.
    - `…pero no detecta ninguna cancelación (recall = 0)` → `recall_tonto` no es `recall_score(y_test, pred_tonto)`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué exactitud y qué recall tiene el modelo tonto?* → **94,0 %** de exactitud y **0 %** de recall.
2. *¿Por qué acierta tanto sin saber nada?* → Porque el **94 % de las reservas no se cancela**: diciendo siempre «no cancela» acierta en todas esas.
3. *¿Qué te dice sobre usar la exactitud con clases raras?* → Que **engaña**: un modelo completamente inútil (no detecta ni una cancelación) saca un 94 %. Con eventos raros hay que mirar recall, precisión, AUC o AP.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
tonto = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
pred_tonto = tonto.predict(X_test)
acc_tonto = accuracy_score(y_test, pred_tonto)
recall_tonto = recall_score(y_test, pred_tonto)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 2. Curva ROC y AUC

#### A3 · Ejemplo — Entrenar el modelo base y sacar sus probabilidades

**Idea.** Para **cada umbral posible** medimos:
- **TPR** (tasa de verdaderos positivos = recall): cancelaciones detectadas,
- **FPR** (tasa de falsos positivos): reservas que se mantienen pero marcamos como riesgo.

La **curva ROC** dibuja TPR frente a FPR. Un modelo inútil va por la diagonal; uno bueno se pega a la esquina superior izquierda. El **AUC** (área bajo la curva) resume: 0,5 = azar; 1 = perfecto. Interpretación intuitiva: *la probabilidad de que una reserva cancelada reciba una puntuación más alta que una que no se cancela*.

- **Qué hace:**
    - `nuevo_modelo().fit(X_train, y_train)` entrena la regresión logística con su preprocesado (`base`).
    - `base.predict_proba(X_test)[:, 1]` da, para cada reserva de test, la probabilidad de cancelar (`proba_base`).
    - El `print` cuenta las reservas de test y cuántas se cancelan.
- **Qué debe salir:**
    - `Reservas de test: 1036 · canceladas: 62`.
- **Si falla:**
    - `NameError: name 'nuevo_modelo' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas reservas hay en test y cuántas canceladas?* → **1036** reservas y **62** canceladas (el 6 %).
2. *¿Qué contiene `proba_base`?* → Una **probabilidad de cancelar** por cada reserva de test (1036 valores entre 0 y 1).
3. *Con solo 62 positivas en test, ¿qué cuidado hay que tener al interpretar las métricas?* → Que son **poco estables**: pocas reservas más o menos detectadas cambian mucho el recall (cada positiva vale 1,6 puntos). Lo retomamos en la reflexión.

</details>

#### A4 · Ejercicio 2 — Calcular la curva ROC y el AUC

- **Qué hace:**
    - `roc_curve(y_test, proba_base)` devuelve tres arrays: la tasa de falsos positivos (`fpr`), la de verdaderos positivos (`tpr`) y los umbrales (que no usamos: `_`). Cada punto es un umbral distinto.
    - `roc_auc_score(y_test, proba_base)` calcula el área bajo esa curva, el AUC.
    - Las líneas que siguen (ya hechas) dibujan la curva y la diagonal del azar.
- **Qué debe salir:**
    - `✔ Ejercicio 2 correcto`, y un gráfico con una curva por encima de la diagonal (el título de la leyenda dice `AUC = 0.77`).
- **Si falla:**
    - `fpr y tpr deben ser la curva ROC completa` → `fpr` y `tpr` no salen de `roc_curve(...)` con `y_test` y `proba_base`.
    - `El AUC de este modelo debería estar entre 0,6 y 0,95` → `auc` no es `roc_auc_score(y_test, proba_base)` (¿has puesto `predict` en vez de las probabilidades?).
    - `NameError: name 'proba_base' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto vale el AUC?* → **≈ 0,77** (0,774).
2. *¿Cómo se interpreta un AUC de 0,77?* → Si tomas una reserva cancelada y otra no al azar, el modelo da **más probabilidad a la cancelada en ≈ 77 % de los casos** (0,5 sería el azar).
3. *¿Qué indica que la curva quede por encima de la diagonal?* → Que el modelo **ordena mejor que el azar**. Por ejemplo, con un FPR de ≈ 0,18 (marcando como riesgo el 18 % de las reservas que se mantienen) detecta ≈ 56–58 % de las cancelaciones.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
fpr, tpr, _ = roc_curve(y_test, proba_base)
auc = roc_auc_score(y_test, proba_base)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 3. Curva precisión–recall (PR) y *average precision*

#### A5 · Ejemplo — Curva precisión–recall

**Problema de la ROC con clases raras:** como el FPR se divide entre muchísimos negativos, la curva puede parecer muy buena aunque **la mayoría de las alarmas sean falsas**. La curva **PR** mira precisión frente a recall, solo con la clase que importa. La línea de referencia no es la diagonal sino la **prevalencia** (el % de positivos): un modelo al azar tiene una precisión igual a la prevalencia. El resumen es la ***average precision*** (AP).

- **Qué hace:**
    - `precision_recall_curve(y_test, proba_base)` da la precisión y el recall de cada umbral; `average_precision_score(...)` los resume en la AP.
    - `prevalencia = y_test.mean()` es la proporción de positivas en test, la referencia del azar.
    - Dibuja la curva PR y una línea horizontal gris en la prevalencia.
- **Qué debe salir:**
    - Un gráfico: la curva empieza alta a la izquierda, cae deprisa y tiende hacia la línea gris (≈ 0,06). Leyenda: `AP = 0.24`.
- **Si falla:**
    - `NameError: name 'proba_base' is not defined` → falta ejecutar el ejemplo del modelo base.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto valen la prevalencia y la AP?* → Prevalencia **≈ 0,06** y AP **≈ 0,24**.
2. *¿Qué representa la línea gris?* → El **azar**: un modelo que marcara reservas al azar tendría una precisión igual a la prevalencia (0,06).
3. *¿Por qué la precisión cae al aumentar el recall?* → Porque para detectar más cancelaciones hay que **marcar más reservas como riesgo**, y entran muchas falsas alarmas: la precisión desciende hacia la prevalencia (0,06), que es lo que obtendrías marcando todas.

</details>

#### A6 · Ejercicio 3 — Cuánto mejora la AP al azar

- **Qué hace:**
    - `average_precision_score(y_test, proba_base)` calcula la AP (`ap_base`).
    - `ap_base / prevalencia` dice cuántas veces es mayor que la precisión del azar.
- **Qué debe salir:**
    - `AP = 0.24: el modelo es 4.0 veces mejor que el azar en precisión.`
    - `✔ Ejercicio 3 correcto`.
- **Si falla:**
    - `assert 0 < ap_base < 1` → `ap_base` no es una AP (¿has usado `roc_auc_score`?).
    - `El modelo debería ser claramente mejor que el azar` → la división está al revés (`prevalencia / ap_base`).
    - `NameError: name 'prevalencia' is not defined` → falta el ejemplo de la curva PR.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto valen `ap_base` y `veces_mejor_que_azar`?* → **≈ 0,24** y **≈ 4,0**.
2. *¿Qué significa «4 veces mejor que el azar»?* → Que, en promedio, las alarmas del modelo acaban siendo ciertas **unas 4 veces más** que si marcáramos reservas al azar (24 % frente a 6 %).
3. *El AUC (0,77) parece bueno y la AP (0,24) parece pobre. ¿Por qué?* → Porque la ROC divide entre los muchísimos negativos (hay unos 2300 en train): unas pocas decenas de falsas alarmas apenas mueven el FPR, pero frente a solo 62 positivas **son la mayoría de las alarmas**. La curva PR sí lo refleja.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
ap_base = average_precision_score(y_test, proba_base)
veces_mejor_que_azar = ap_base / prevalencia
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 4. Tres remedios para el desbalance

#### A7 · Ejercicio 4 — Pesos por clase

1. **Pesos por clase** (`class_weight="balanced"`): un error en la clase rara cuenta más al entrenar.
2. **Submuestreo** de la clase mayoritaria: entrenar con menos negativos (más equilibrado, pero se tira información). *(También existe el sobremuestreo y técnicas como SMOTE, que requieren otra librería y no usamos aquí).*
3. **Ajustar el umbral** de decisión (ya lo hicimos en la práctica anterior): no cambia el modelo, cambia la decisión.

Regla clave: **el remuestreo solo se aplica al conjunto de entrenamiento**, nunca al de test.

- **Qué hace:**
    - `nuevo_modelo(class_weight="balanced")` crea el pipeline con una regresión logística que pondera cada clase al revés de su frecuencia; `.fit(X_train, y_train)` lo ajusta.
    - `recall_score(y_test, modelo.predict(X_test))` mide el recall en test (umbral 0,5) de `base` y de `con_pesos`; `precision_score` mide la precisión del ponderado.
- **Qué debe salir:**
    - `Recall: 3% → 73% · precisión del ponderado: 14%`.
    - `✔ Ejercicio 4 correcto`.
- **Si falla:**
    - `Con pesos por clase el recall debería subir` → `con_pesos` se ha creado sin `class_weight="balanced"`, o has medido el recall con otro conjunto.
    - `El modelo ponderado debería detectar más de la mitad de las cancelaciones` → igual: el recall del ponderado es < 0,5.
    - `NameError: name 'base' is not defined` → falta el ejemplo del modelo base.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cómo cambia el recall y qué precisión tiene el modelo ponderado?* → El recall pasa de **3 % a 73 %** y la precisión del ponderado es **14 %**.
2. *¿Qué hace `class_weight="balanced"`?* → Da a cada clase un peso inversamente proporcional a su frecuencia: aquí, un error en una cancelada **cuenta unas 16 veces más** que uno en una no cancelada (hay ≈ 16 veces menos).
3. *¿Qué se paga a cambio de detectar el 73 %?* → La precisión: solo el 14 % de las alarmas son ciertas, es decir, **86 de cada 100 son falsas**. El modelo base apenas daba alarmas (3 en test) y por eso fallaba pocas, pero detectaba solo 2 de las 62.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
con_pesos = nuevo_modelo(class_weight="balanced").fit(X_train, y_train)
recall_base = recall_score(y_test, base.predict(X_test))
recall_pesos = recall_score(y_test, con_pesos.predict(X_test))
precision_pesos = precision_score(y_test, con_pesos.predict(X_test))
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A8 · Ejercicio 5 — Submuestreo de la clase mayoritaria

- **Qué hace:**
    - `pos_tr` y `neg_tr` separan las reservas de train canceladas y no canceladas.
    - `resample(neg_tr, replace=False, n_samples=2 * len(pos_tr), random_state=SEMILLA)` elige al azar, **sin repetir**, el doble de negativas que de positivas.
    - `pd.concat([pos_tr, neg_sub])` forma `X_sub`; `y_sub` es su respuesta (unos para las positivas y ceros para las negativas); `con_submuestreo` es un modelo entrenado solo con esas filas.
- **Qué debe salir:**
    - `✔ Ejercicio 5 correcto`. `X_sub` tiene 435 filas (145 positivas y 290 negativas) frente a las 2417 de `X_train`; el recall en test del modelo es ≈ 0,45.
- **Si falla:**
    - `Debe haber el doble de negativas que de positivas` → `n_samples` no es `2 * len(pos_tr)`, o `y_sub` no corresponde a `X_sub`.
    - `El submuestreo reduce mucho el tamaño del entrenamiento` → no has reducido las negativas (¿`replace=True` o todas?).
    - `El submuestreo debería subir el recall` → has remuestreado test, o has entrenado `con_submuestreo` con `X_train`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas filas tiene `X_sub` y cómo se reparten?* → **435**: 145 positivas y 290 negativas (frente a las 2417 de `X_train`).
2. *¿Por qué el recall del submuestreo (≈ 0,45) es menor que el de los pesos (0,73)?* → Porque aquí la clase rara pesa **2 a 1** (hay el doble de negativas), mientras que con `balanced` pesa como si fueran **1 a 1**. Con 2 a 1 el modelo es menos «alarmista».
3. *¿Qué inconveniente tiene el submuestreo?* → Se **tira información**: de 2272 negativas de train solo se usan 290. Por eso hay que hacerlo solo con train y, si se puede, repetirlo con otras muestras.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
pos_tr = X_train[y_train == 1]
neg_tr = X_train[y_train == 0]
neg_sub = resample(neg_tr, replace=False, n_samples=2 * len(pos_tr), random_state=SEMILLA)
X_sub = pd.concat([pos_tr, neg_sub])
y_sub = pd.Series([1] * len(pos_tr) + [0] * len(neg_sub), index=X_sub.index)
con_submuestreo = nuevo_modelo().fit(X_sub, y_sub)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 5. Ajustar el umbral con la curva PR

#### A9 · Ejercicio 6 — Umbral para detectar al menos el 70 %

- **Qué hace:**
    - `cross_val_predict(..., method="predict_proba")` da, para cada reserva de train, una probabilidad calculada por un modelo que no la ha visto (`proba_cv`).
    - `precision_recall_curve(y_train, proba_cv)` devuelve la precisión, el recall y los umbrales. Ojo: `precision` y `recall` tienen un elemento más que `umbrales`, por eso se usa `r_cv[:-1]`.
    - `u_cv[r_cv[:-1] >= 0.70].max()` es el **mayor** umbral cuyo recall es ≥ 0,70 (el que pierde menos precisión).
    - Con ese umbral se predice en test (`proba_base >= umbral_70`) y se mide recall y precisión.
- **Qué debe salir:**
    - `Umbral 0.048: recall en test 82%, precisión 12%`.
    - `✔ Ejercicio 6 correcto`.
- **Si falla:**
    - `Para alcanzar recall 0,70 con clases raras el umbral debe ser bajo` → has cogido el menor umbral con recall ≥ 0,70 (casi 0) o el umbral por defecto: usa `.max()`.
    - `En test el recall debería aproximarse al objetivo del 70 %` → has calculado `umbral_70` con test o has aplicado el umbral al modelo equivocado.
    - `IndexError`/forma distinta → `recall` tiene un elemento más que `umbrales`: recórtalo con `[:-1]`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué umbral sale y qué recall y precisión da en test?* → **0,048**, con recall **82 %** y precisión **12 %**.
2. *¿Por qué el umbral es tan bajo (0,048) y no 0,5?* → Porque con solo un 6 % de positivas, las probabilidades que da el modelo son pequeñas: para detectar el 70 % hay que marcar como riesgo reservas con una probabilidad de solo ≈ 5 %.
3. *El objetivo era el 70 % y en test sale el 82 %. ¿Debemos fiarnos?* → No del todo: hay solo **62 positivas** en test, así que el recall fluctúa mucho (cada una vale 1,6 puntos). El umbral se eligió bien (con validación cruzada en train); la diferencia es parte del azar.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
cv = StratifiedKFold(5, shuffle=True, random_state=SEMILLA)
proba_cv = cross_val_predict(nuevo_modelo(), X_train, y_train, cv=cv, method="predict_proba")[:, 1]
p_cv, r_cv, u_cv = precision_recall_curve(y_train, proba_cv)
umbral_70 = float(u_cv[r_cv[:-1] >= 0.70].max())
pred_70 = (proba_base >= umbral_70).astype(int)
recall_test_70 = recall_score(y_test, pred_70)
precision_test_70 = precision_score(y_test, pred_70)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 6. Mini-reto: comparativa de estrategias

#### A10 · Mini-reto — Comparativa de las tres estrategias

- **Qué hace:**
    - `modelos` es un diccionario con las tres estrategias: el modelo base, el ponderado y el entrenado con submuestreo.
    - Para cada una, calcula en test el AUC y la AP con las **probabilidades** (`predict_proba`) y el recall y la precisión con las **clases** (`predict`, umbral 0,5).
    - `pd.DataFrame(filas).T.round(3)` los junta en una tabla; `comparativa["AP"].idxmax()` da el nombre de la estrategia con mayor AP.
- **Qué debe salir:**
    - `✔ Mini-reto correcto` y la tabla: `base` AUC 0,774 · AP 0,242 · recall 0,032 · precisión 0,667; `pesos` 0,770 · 0,245 · 0,726 · 0,136; `submuestreo` 0,752 · 0,241 · 0,452 · 0,172. `ganadora_ap` vale `'pesos'`.
- **Cómo leerlo:**
    - Observa: **el AUC y la AP cambian poco entre estrategias** (el modelo «ordena» parecido); lo que cambia mucho es el **punto de trabajo** (recall frente a precisión con umbral 0,5). Los pesos y el submuestreo son, en el fondo, formas indirectas de mover el umbral.
- **Si falla:**
    - `assert list(comparativa.index) == ["base", "pesos", "submuestreo"]` → falta `.T` o los nombres/orden no coinciden.
    - `assert {"AUC", "AP", "recall", "precision"} <= set(comparativa.columns)` → alguna columna tiene otro nombre.
    - `assert ganadora_ap in comparativa.index` → `idxmax()` debe aplicarse a la columna `AP`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué estrategia gana en AP y cuánta ventaja tiene?* → **`pesos`** (0,245), pero la ventaja es **mínima**: 0,242 la base y 0,241 el submuestreo. En la práctica, empatan.
2. *¿Qué cambia mucho entre las tres estrategias?* → El **recall**: 0,03 (base), 0,73 (pesos) y 0,45 (submuestreo), y con él la precisión (0,67, 0,14 y 0,17). Cambia el punto de trabajo, no la capacidad de ordenar.
3. *¿Por qué el AUC y la AP casi no cambian?* → Porque las tres versiones del modelo **ordenan las reservas casi igual**; los pesos y el submuestreo solo desplazan las probabilidades hacia arriba, como si bajáramos el umbral.

</details>

## 7. Para reflexionar

1. Con un 6 % de positivos, ¿por qué la ROC puede parecer optimista y la PR no?
2. Si tu objetivo es no perder ninguna cancelación importante, ¿qué métrica vigilas? ¿Y si cada alarma cuesta una llamada de un comercial?
3. ¿Por qué el submuestreo solo se hace con datos de entrenamiento?
4. Con solo ≈ 60 positivas en test, ¿cuánto te fías de un recall del 70 %? ¿Cómo darías una idea de la incertidumbre?

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. Porque el FPR se divide entre los muchísimos negativos (≈ 970 en test): decenas de falsas alarmas apenas lo mueven. Pero frente a solo 62 positivas esas falsas alarmas **son la mayoría de las alarmas**, y la precisión (la PR) lo refleja: la PR parte de la prevalencia (0,06) como azar y exige más.
2. Para no perder cancelaciones, el **recall**. Si cada alarma cuesta una llamada, hay que vigilar también la **precisión** (o el coste total, como en la práctica anterior) y mirar la curva PR para elegir un umbral que equilibre ambos.
3. Porque el test debe parecerse a la **realidad en la que se usará el modelo** (con su 6 % real de cancelaciones). Si lo remuestreáramos, mediríamos un mundo que no existe y las métricas saldrían falseadas. El remuestreo es un truco para **entrenar**, no para evaluar.
4. Poco: con 62 positivas, un recall del 70 % tiene un margen de error de unos ±11 puntos (≈ de 59 % a 81 %). Se daría un **intervalo de confianza** (binomial o por *bootstrap*) y se repetiría con validación cruzada, que da más positivas para evaluar.

</details>

## 8. Resumen: qué has aprendido
- Con clases raras la **exactitud engaña**; mira recall, precisión, AUC y AP.
- **ROC/AUC** mide la capacidad de ordenar; **PR/AP** es más exigente con clases raras.
- Remedios: pesos por clase, submuestreo (solo en train) y, sobre todo, un **umbral** ajustado al negocio.